In [1]:
import sys
sys.path.append("../..")

In [2]:
from syntax_tokenizer import SyntaxTokenizer
from model import ModelConfig, LlamaModel
from train import TrainerConfig, SimpleDataLoader, Trainer

In [3]:
with open("../../data/complete_shakespeare.txt") as f:
    text = f.read()

In [4]:
texts = text.split("\n\n\n\n")

In [5]:
tokenizer = SyntaxTokenizer()
tokenizer.train(texts)

153it [00:58,  2.63it/s]


In [6]:
tokenizer.vocab_size

12853

In [7]:
model_config = ModelConfig(
    is_causal=False,
    vocab_size=tokenizer.vocab_size,
    d_model=576,
    d_head=64,
    d_mlp_proj=1536,
    n_layers=30,
    n_kv_heads=3,
    n_attn_heads=9,
    rms_norm_eps=1e-5,
    initializer_range=0.02,
    rope_theta=100000.0,
    padding_idx=tokenizer.data.pad_token_id
)

In [8]:
train_config = TrainerConfig(
    is_causal=False,
    mask_ratio=0.15,
    per_device_train_batch_size=8,
    max_seq_len=512,
    num_epochs=8,
    eval_interval_steps=25,
    learning_rate=4e-3,
    grad_clip_norm=1.0,
    val_size=0.05,
    log_dir="runs/shakespeare_bidirectional",
    warmup_ratio=0.1
)

In [9]:
n = 100
size_patch = len(text) // 100
texts_equal = [text[i:i+size_patch] for i in range(n+1)]

In [10]:
model = LlamaModel(model_config)
dataloader = SimpleDataLoader(train_config, tokenizer, texts=texts_equal)
trainer = Trainer(train_config, model, tokenizer)

Total tokens                   | 1,341,886
Num Trainable Params           | 121,010,112
Train device                   | cuda, NVIDIA GeForce RTX 3090, N=1
Training precision             | torch.bfloat16
Flash Attention                | True
torch.compile()                | True
DistributedDataParallel        | False
Batch size                     | 614




In [11]:
trainer.train(dataloader)

Training steps                 | 2,496 
Step: 0, Training Loss: 9.46608, LR: 0.0002000, Tokens/sec: 383.46
Step: 1, Training Loss: 8.81984, LR: 0.0002153, Tokens/sec: 442.23
Step: 2, Training Loss: 8.06887, LR: 0.0002305, Tokens/sec: 99609.83
Step: 3, Training Loss: 7.65565, LR: 0.0002458, Tokens/sec: 106086.87
Computing Eval loss, steps: 17
Step: 3, Eval Loss: 7.55380
Step: 4, Training Loss: 7.42727, LR: 0.0002610, Tokens/sec: 101374.58
Step: 5, Training Loss: 7.20034, LR: 0.0002763, Tokens/sec: 104964.06
Step: 6, Training Loss: 7.21959, LR: 0.0002916, Tokens/sec: 105527.57
Step: 7, Training Loss: 7.12729, LR: 0.0003068, Tokens/sec: 111455.05
Step: 8, Training Loss: 6.75161, LR: 0.0003221, Tokens/sec: 112266.54
Step: 9, Training Loss: 6.72743, LR: 0.0003373, Tokens/sec: 111972.19
Step: 10, Training Loss: 6.31037, LR: 0.0003526, Tokens/sec: 106414.95
Step: 11, Training Loss: 6.11217, LR: 0.0003679, Tokens/sec: 109414.52
Step: 12, Training Loss: 5.73510, LR: 0.0003831, Tokens/sec: 11111

In [13]:
input_text = """
ALL. Content, content.
""".strip()

input_ids = tokenizer([input_text], return_tensors="pt")['input_ids'].to(trainer.device)
idx = model.generate(input_ids, temperature=2, top_k=500, max_new_tokens=32)
#print(tokenizer.batch_decode(idx)[0])
print(idx[0])

tensor([ 6149,  1528,  6164,   772,  5842,  1528,  7829,     1,  9607,  9915,
            1,  5870, 10923,  9622,  6234,  6079,  9873,  6639,  5998,  4413,
         8347,  9593,  7843,  7843,  9635, 10448,  7857,  9635,  8347,  2537,
         4513,  6723,  7843,  7325,  7325,  7843,  7843,  7843],
       device='cuda:0')
